# Stage 2: real GPU workloads + NVML telemetry (Colab)

**Before you start:** Runtime > Change runtime type > pick a GPU (T4 is fine). Power limit cannot be changed on Colab, so this stage only varies **batch size** and **precision**; the current limit is read and logged.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,power.draw,power.limit,clocks.sm,utilization.gpu --format=csv

## 1. Get the code
Edit `REPO_URL`. For a private repo use `https://<TOKEN>@github.com/<owner>/<repo>.git`, or upload the project zip and `!unzip` it instead.

In [ ]:
REPO_URL = "https://github.com/<owner>/AI-workload-optimisation.git"  # <-- edit
import os
if not os.path.exists("repo"):
    !git clone {REPO_URL} repo
%cd repo

In [ ]:
!pip install -q nvidia-ml-py transformers datasets pyyaml

## 2. Smoke test (about a minute)
Checks that NVML, PyTorch and logging all work before the long sweep. Power numbers from such short runs are not meaningful.

In [ ]:
!python scripts/run_stage2.py --smoke --out data/stage2_smoke.csv

## 3. Full sweep
Defaults come from `configs/sweep_config.yaml` (`sweep` and `stage2` sections). Each run is appended to the CSV immediately, so a disconnect loses nothing. Expect roughly 15-30 minutes on a T4.

In [ ]:
!python scripts/run_stage2.py

## 4. Analyse

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

CSV = "data/stage2_gpu_results.csv"
BASE_BS, BASE_PREC, MAX_SLOWDOWN = 32, "fp32", 1.05   # same baseline / constraint as Stage 1

df = pd.read_csv(CSV)
med = (df.groupby(["workload", "precision", "batch_size"], as_index=False)
         .agg(total_samples=("total_samples", "first"),
              runtime_s=("runtime_s", "median"), avg_power_w=("avg_power_w", "median"),
              energy_j=("energy_j", "median"), throughput_sps=("throughput_sps", "median"),
              runtime_cv_pct=("runtime_s", lambda s: 100 * s.std(ddof=0) / s.mean())))
med["energy_per_1k_j"] = 1000 * med.energy_j / med.total_samples

# --- plots: one row per workload ---------------------------------------------------
workloads = list(med.workload.unique())
fig, axes = plt.subplots(len(workloads), 3, figsize=(15, 4.2 * len(workloads)), squeeze=False)
for r, w in enumerate(workloads):
    for c, (col, label) in enumerate([("energy_per_1k_j", "Energy per 1000 samples (J)"),
                                      ("runtime_s", "Runtime per run (s)"),
                                      ("avg_power_w", "Average power (W)")]):
        ax = axes[r][c]
        for prec, g in med[med.workload == w].groupby("precision"):
            g = g.sort_values("batch_size")
            ax.plot(g.batch_size, g[col], marker="o", label=prec)
        ax.set_xscale("log", base=2); ax.set_xlabel("Batch size"); ax.set_ylabel(label)
        ax.set_title(w); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig("data/stage2_overview.png", dpi=150); plt.show()

# --- the Stage 1 objective applied to the real measurements ------------------------
for w, g in med.groupby("workload"):
    base = g[(g.batch_size == BASE_BS) & (g.precision == BASE_PREC)].iloc[0]
    g = g.assign(slowdown=g.runtime_s / base.runtime_s, energy_ratio=g.energy_j / base.energy_j)
    g["valid"] = g.slowdown <= MAX_SLOWDOWN
    best = g[g.valid].sort_values("energy_j").iloc[0]
    print(f"\n{w}: baseline bs={BASE_BS}/{BASE_PREC} -> {base.runtime_s:.2f}s, {base.energy_j:.0f}J")
    print(f"  best valid: bs={int(best.batch_size)}/{best.precision} -> {best.runtime_s:.2f}s, "
          f"{best.energy_j:.0f}J ({100 * (1 - best.energy_ratio):.1f}% less energy, "
          f"{100 * (best.slowdown - 1):+.1f}% runtime)")
    display(g.sort_values("energy_j")[["batch_size", "precision", "runtime_s", "avg_power_w", "energy_j",
                                       "slowdown", "energy_ratio", "runtime_cv_pct", "valid"]].round(3))

## 5. Download results
Colab storage is wiped when the session ends.

In [ ]:
from google.colab import files
for f in ["data/stage2_gpu_results.csv", "data/stage2_gpu_results.env.json", "data/stage2_overview.png"]:
    files.download(f)